In [ ]:
# Importa as bibliotecas necessárias para realizar o Grid Search com TF-IDF e Regressão Logística.

import pandas as pd

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from registro_resultados import registrar_resultado

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from registro_resultados import registrar_resultado

In [ ]:
# Define o arquivo, as colunas e os parâmetros da validação cruzada utilizada no Grid Search.

ARQUIVO_TREINO = "../train.xlsx"

COLUNA_TEXTO = "resp_text"
COLUNA_CLASSE = "clarity"

RANDOM_STATE = 42
N_SPLITS = 5


In [ ]:
# Carrega os dados, trata valores ausentes e separa os textos das classes.

df = pd.read_excel(ARQUIVO_TREINO)

df[COLUNA_TEXTO] = (
    df[COLUNA_TEXTO]
    .fillna("")
    .astype(str)
)

print("Dimensões:", df.shape)

print("\nColunas:")
print(df.columns.tolist())

print("\nDistribuição das classes:")
print(df[COLUNA_CLASSE].value_counts())

X = df[COLUNA_TEXTO]
y = df[COLUNA_CLASSE]


In [ ]:
# Cria a validação cruzada estratificada que será utilizada para comparar as configurações do modelo.

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

In [ ]:
# Cria o pipeline que combina a representação TF-IDF com o classificador de Regressão Logística.

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("logistic", LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

In [ ]:
# Define as combinações de n-gramas e regularização que serão avaliadas pelo Grid Search.

param_grid = {
    "tfidf__ngram_range": [
        (1, 1),
        (1, 2)
    ],
    "logistic__C": [
        0.5,
        1.0,
        2.0
    ]
}

In [ ]:
# Configura o Grid Search para comparar as combinações utilizando acurácia como critério principal e F1 macro como métrica complementar.

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "f1_macro": "f1_macro"
    },
    refit="accuracy",
    n_jobs=2,
    pre_dispatch=2,
    return_train_score=True,
    error_score="raise"
)

In [ ]:
# Executa o Grid Search sobre todo o conjunto de dados utilizando a validação cruzada configurada.

print("==============================")
print("INICIANDO GRID SEARCH")
print("==============================")

grid_search.fit(X, y)

In [ ]:
# Exibe os parâmetros da configuração selecionada e sua acurácia média na validação cruzada.

print("==============================")
print("MELHOR RESULTADO")
print("==============================")

print("Melhores parâmetros:")
print(grid_search.best_params_)

melhor_acuracia = float(
    grid_search.best_score_
)

print(f"\nMelhor acurácia média: {melhor_acuracia:.4f}")

In [ ]:
# Extrai os resultados completos do Grid Search e seleciona as métricas relevantes para análise.

resultados = pd.DataFrame(
    grid_search.cv_results_
)

resultados = resultados[
    [
        "param_tfidf__ngram_range",
        "param_logistic__C",
        "mean_train_accuracy",
        "std_train_accuracy",
        "mean_test_accuracy",
        "std_test_accuracy",
        "mean_train_f1_macro",
        "mean_test_f1_macro"
    ]
].copy()

In [ ]:
# Renomeia as colunas dos resultados para nomes mais claros e ordena as configurações pela acurácia de validação.

resultados = resultados.rename(columns={
    "param_tfidf__ngram_range": "ngram_range",
    "param_logistic__C": "C",
    "mean_train_accuracy": "accuracy_treino_media",
    "std_train_accuracy": "desvio_accuracy_treino",
    "mean_test_accuracy": "accuracy_validacao_media",
    "std_test_accuracy": "desvio_accuracy_validacao",
    "mean_train_f1_macro": "f1_treino_medio",
    "mean_test_f1_macro": "f1_validacao_medio"
})

resultados = resultados.sort_values(
    by="accuracy_validacao_media",
    ascending=False
).reset_index(drop=True)

print(resultados.to_string(index=False))

In [ ]:
# Calcula a diferença entre a acurácia média de treinamento e validação como indicador simples de possível overfitting.

resultados["gap_accuracy"] = (
    resultados["accuracy_treino_media"]
    - resultados["accuracy_validacao_media"]
)

print("==============================")
print("GAP TREINO - VALIDAÇÃO")
print("==============================")

print(
    resultados[
        [
            "ngram_range",
            "C",
            "accuracy_treino_media",
            "accuracy_validacao_media",
            "gap_accuracy"
        ]
    ].to_string(index=False)
)

In [ ]:
# Monta o registro consolidado da melhor configuração encontrada pelo Grid Search.

melhor_linha = resultados.iloc[0]

resultado_grid = {
    "experimento": "Grid Search - TF-IDF + Regressão Logística",
    "representacao": "TF-IDF",
    "modelo": "Regressão Logística",
    "acuracia_media": float(grid_search.best_score_),
    "f1_macro_medio": float(
        melhor_linha["f1_validacao_medio"]
    ),
    "gap_accuracy": float(
        melhor_linha["gap_accuracy"]
    ),
    "melhores_parametros": {
        "tfidf__ngram_range": tuple(
            int(valor)
            for valor in grid_search.best_params_[
                "tfidf__ngram_range"
            ]
        ),
        "logistic__C": float(
            grid_search.best_params_["logistic__C"]
        )
    }
}

print("Resultado:")
print(resultado_grid)

In [ ]:
# Registra a melhor configuração encontrada na tabela consolidada de experimentos.

registrar_resultado(resultado_grid)

In [ ]:
# Converte as colunas numéricas dos resultados para tipos float antes do salvamento da tabela.

colunas_numericas = [
    "C",
    "accuracy_treino_media",
    "desvio_accuracy_treino",
    "accuracy_validacao_media",
    "desvio_accuracy_validacao",
    "f1_treino_medio",
    "f1_validacao_medio",
    "gap_accuracy"
]

for coluna in colunas_numericas:
    resultados[coluna] = resultados[coluna].astype(float)

In [ ]:
# Salva a tabela completa das configurações testadas pelo Grid Search em um arquivo CSV.

resultados.to_csv(
    "resultados_grid_baseline.csv",
    index=False
)

print(
    "Tabela salva em: resultados_grid_baseline.csv"
)